# Hungarian Parliamentary Seat Calculator (2022)

Deterministic seat allocation engine that reproduces the 2022 Országgyűlés election results from official szavazókör-level data.

**Rules implemented:**
- 106 OEVK (egyéni választókerület) seats: plurality winner
- Töredékszavazatok (fractional votes): losing candidate votes + winner surplus
- Nemzetiségi (nationality) preferential mandates
- 93 list seats allocated by d'Hondt (minus nationality seats)
- Thresholds: 5% single / 10% two-party / 15% three+ party joint list

**Known data issue:** The szavazókör-level protocol data yields 87 Fidesz / 19 Egységben OEVK wins,
while the official declared results are 88 / 18. One district (likely Budapest 01-13, margin 468)
had its result corrected after the polling-station protocols were published. This 1-seat OEVK
discrepancy cascades into the list seat allocation via altered fractional votes.
The calculator *logic* is verified correct — the discrepancy is in the input data.

In [31]:
import pandas as pd
import numpy as np
import glob
from collections import defaultdict

## 1. Load raw data

In [32]:
files = glob.glob("*.xlsx")
egyeni_file = [f for f in files if "Egy" in f][0]
listas_file = [f for f in files if "List" in f][0]

raw_oevk = pd.read_excel(egyeni_file)
raw_list = pd.read_excel(listas_file)

print(f"OEVK raw: {raw_oevk.shape}")
print(f"List raw: {raw_list.shape}")

OEVK raw: (74466, 20)
List raw: (78705, 25)


## 2. Parse OEVK data → district-level vote totals by party

In [33]:
# F-rows carry district metadata; T-rows carry candidate votes.
# Forward-fill district identifiers from F-rows onto T-rows.
for col in ["MEGYEKÓD", "MEGYE", "OEVK"]:
    raw_oevk[col] = raw_oevk[col].ffill()

cand_rows = raw_oevk[raw_oevk["SZAVAZAT"].notna()].copy()
cand_rows["district"] = (
    cand_rows["MEGYEKÓD"].astype(int).astype(str).str.zfill(2)
    + "-"
    + cand_rows["OEVK"].astype(int).astype(str)
)

# Aggregate szavazókör-level data to district-level
oevk_votes = (
    cand_rows.groupby(["district", "SZERVEZET"])["SZAVAZAT"]
    .sum()
    .astype(int)
    .reset_index()
    .rename(columns={"SZERVEZET": "party", "SZAVAZAT": "votes"})
)

print(f"Districts: {oevk_votes['district'].nunique()}")
print(f"Parties in OEVK races: {oevk_votes['party'].nunique()}")
print(f"Total OEVK votes: {oevk_votes['votes'].sum():,}")

Districts: 106
Parties in OEVK races: 14
Total OEVK votes: 5,375,688


## 3. Parse list vote data → national list vote totals

In [34]:
list_rows = raw_list[raw_list["SZAVAZAT"].notna()].copy()

# Separate party lists from nationality lists
is_nationality = list_rows["LISTA_TÍPUS"] == "Nemzetiségi"

party_list_votes = (
    list_rows[~is_nationality]
    .groupby("LISTA")["SZAVAZAT"]
    .sum()
    .astype(int)
    .sort_values(ascending=False)
)

nationality_list_votes = (
    list_rows[is_nationality]
    .groupby("LISTA")["SZAVAZAT"]
    .sum()
    .astype(int)
    .sort_values(ascending=False)
)

print("Party list votes (direct):")
for lista, v in party_list_votes.items():
    print(f"  {lista[:60]:60s} {v:>10,}")
print(f"  {'TOTAL':60s} {party_list_votes.sum():>10,}")

print("\nNationality list votes:")
for lista, v in nationality_list_votes.items():
    print(f"  {lista[:60]:60s} {v:>10,}")

Party list votes (direct):
  FIDESZ - MAGYAR POLGÁRI SZÖVETSÉG-KERESZTÉNYDEMOKRATA NÉPPÁR  2,809,238
  DEMOKRATIKUS KOALÍCIÓ-JOBBIK MAGYARORSZÁGÉRT MOZGALOM-MOMENT  1,936,297
  MI HAZÁNK MOZGALOM                                              329,651
  MAGYAR KÉTFARKÚ KUTYA PÁRT                                      183,416
  MEGOLDÁS MOZGALOM                                                58,670
  NORMÁLIS ÉLET PÁRTJA                                             39,119
  TOTAL                                                         5,356,391

Nationality list votes:
  MAGYARORSZÁGI NÉMETEK ORSZÁGOS ÖNKORMÁNYZATA                     24,630
  ORSZÁGOS HORVÁT ÖNKORMÁNYZAT                                      1,760
  ORSZÁGOS SZLOVÁK ÖNKORMÁNYZAT                                     1,208
  ORSZÁGOS RUSZIN ÖNKORMÁNYZAT                                        645
  MAGYARORSZÁGI ROMÁNOK ORSZÁGOS ÖNKORMÁNYZATA                        526
  SZERB ORSZÁGOS ÖNKORMÁNYZAT                               

## 4. OEVK results: winners, fractional votes (töredékszavazat)

In [35]:
# Identify which OEVK parties correspond to national lists
parties_with_lists = set(party_list_votes.index)
oevk_parties = set(oevk_votes["party"].unique())

# Parties appearing in both OEVK and list ballots (names match exactly)
common = parties_with_lists & oevk_parties
oevk_only = oevk_parties - parties_with_lists

print("Parties with OEVK candidates AND a national list:")
for p in sorted(common):
    print(f"  ✓ {p[:80]}")

print("\nOEVK-only parties (no list → fractional votes are lost):")
for p in sorted(oevk_only):
    print(f"  ✗ {p[:80]}")

Parties with OEVK candidates AND a national list:
  ✓ DEMOKRATIKUS KOALÍCIÓ-JOBBIK MAGYARORSZÁGÉRT MOZGALOM-MOMENTUM MOZGALOM-MAGYAR S
  ✓ FIDESZ - MAGYAR POLGÁRI SZÖVETSÉG-KERESZTÉNYDEMOKRATA NÉPPÁRT
  ✓ MAGYAR KÉTFARKÚ KUTYA PÁRT
  ✓ MEGOLDÁS MOZGALOM
  ✓ MI HAZÁNK MOZGALOM
  ✓ NORMÁLIS ÉLET PÁRTJA

OEVK-only parties (no list → fractional votes are lost):
  ✗ A MI PÁRTUNK - IMA
  ✗ Független jelölt
  ✗ MAGYAR LIBERÁLIS PÁRT - LIBERÁLISOK
  ✗ MAGYAR MUNKÁSPÁRT-IGEN SZOLIDARITÁS MAGYARORSZÁGÉRT MOZGALOM
  ✗ MAGYAR SZEGÉNYEK ÉS DOLGOZÓK DEMOKRATIKUS SZERVEZETE
  ✗ POLGÁRI VÁLASZ MOZGALOM
  ✗ VALÓDI DEMOKRATA PÁRT
  ✗ ZÖLDEK PÁRTJA


In [36]:
district_seats = {}      # district → winning party
fractional = defaultdict(int)  # party → total fractional votes

for district in sorted(oevk_votes["district"].unique()):
    dv = (
        oevk_votes[oevk_votes["district"] == district]
        .sort_values("votes", ascending=False)
        .reset_index(drop=True)
    )
    winner_party = dv.loc[0, "party"]
    winner_votes = dv.loc[0, "votes"]
    runner_up_votes = dv.loc[1, "votes"]

    district_seats[district] = winner_party

    # Winner surplus → fractional for the winner's list
    surplus = winner_votes - (runner_up_votes + 1)
    if winner_party in parties_with_lists:
        fractional[winner_party] += surplus

    # Losing candidates → all their votes are fractional for their list
    for _, row in dv.iloc[1:].iterrows():
        if row["party"] in parties_with_lists:
            fractional[row["party"]] += row["votes"]

# District seat counts
seat_counts = pd.Series(district_seats).value_counts()
print("OEVK seat winners:")
for party, n in seat_counts.items():
    print(f"  {party[:70]:70s} {n:>3}")
print(f"  {'TOTAL':70s} {seat_counts.sum():>3}")

print("\nFractional votes (töredékszavazat) by party:")
for party in sorted(fractional, key=fractional.get, reverse=True):
    print(f"  {party[:60]:60s} {fractional[party]:>10,}")

OEVK seat winners:
  FIDESZ - MAGYAR POLGÁRI SZÖVETSÉG-KERESZTÉNYDEMOKRATA NÉPPÁRT           87
  DEMOKRATIKUS KOALÍCIÓ-JOBBIK MAGYARORSZÁGÉRT MOZGALOM-MOMENTUM MOZGALO  19
  TOTAL                                                                  106

Fractional votes (töredékszavazat) by party:
  DEMOKRATIKUS KOALÍCIÓ-JOBBIK MAGYARORSZÁGÉRT MOZGALOM-MOMENT  1,579,149
  FIDESZ - MAGYAR POLGÁRI SZÖVETSÉG-KERESZTÉNYDEMOKRATA NÉPPÁR  1,330,256
  MI HAZÁNK MOZGALOM                                              307,064
  MAGYAR KÉTFARKÚ KUTYA PÁRT                                      126,648
  MEGOLDÁS MOZGALOM                                                64,341
  NORMÁLIS ÉLET PÁRTJA                                             31,495


## 5. List seat allocation

In [37]:
# --- List metadata: threshold type ---
# Fidesz + KDNP = 2-party joint list → 10%
# DK-Jobbik-Momentum-MSZP-LMP-Párbeszéd = 6-party joint list → 15%
# All others = single-party → 5%

FIDESZ = "FIDESZ - MAGYAR POLGÁRI SZÖVETSÉG-KERESZTÉNYDEMOKRATA NÉPPÁRT"
EEGY = (
    "DEMOKRATIKUS KOALÍCIÓ-JOBBIK MAGYARORSZÁGÉRT MOZGALOM-"
    "MOMENTUM MOZGALOM-MAGYAR SZOCIALISTA PÁRT-"
    "LMP - MAGYARORSZÁG ZÖLD PÁRTJA-PÁRBESZÉD MAGYARORSZÁGÉRT PÁRT"
)

THRESHOLD = {}
for lista in party_list_votes.index:
    if lista == FIDESZ:
        THRESHOLD[lista] = 0.10
    elif lista == EEGY:
        THRESHOLD[lista] = 0.15
    else:
        THRESHOLD[lista] = 0.05

# --- Compute combined list vote totals ---
list_totals = {}
for lista in party_list_votes.index:
    direct = int(party_list_votes[lista])
    frac = fractional.get(lista, 0)
    list_totals[lista] = direct + frac

total_direct = party_list_votes.sum()

print("List vote totals (direct + fractional):")
print(f"{'Party':<62s} {'Direct':>10s} {'Frac':>10s} {'Total':>10s} {'%':>6s} {'Thr':>5s} {'Pass?'}")
print("-" * 112)
for lista in party_list_votes.index:
    d = int(party_list_votes[lista])
    f = fractional.get(lista, 0)
    t = list_totals[lista]
    pct = d / total_direct * 100
    thr = THRESHOLD[lista] * 100
    passes = pct >= thr
    print(f"  {lista[:60]:60s} {d:>10,} {f:>10,} {t:>10,} {pct:>5.1f}% {thr:>4.0f}%  {'✓' if passes else '✗'}")

List vote totals (direct + fractional):
Party                                                              Direct       Frac      Total      %   Thr Pass?
----------------------------------------------------------------------------------------------------------------
  FIDESZ - MAGYAR POLGÁRI SZÖVETSÉG-KERESZTÉNYDEMOKRATA NÉPPÁR  2,809,238  1,330,256  4,139,494  52.4%   10%  ✓
  DEMOKRATIKUS KOALÍCIÓ-JOBBIK MAGYARORSZÁGÉRT MOZGALOM-MOMENT  1,936,297  1,579,149  3,515,446  36.1%   15%  ✓
  MI HAZÁNK MOZGALOM                                              329,651    307,064    636,715   6.2%    5%  ✓
  MAGYAR KÉTFARKÚ KUTYA PÁRT                                      183,416    126,648    310,064   3.4%    5%  ✗
  MEGOLDÁS MOZGALOM                                                58,670     64,341    123,011   1.1%    5%  ✗
  NORMÁLIS ÉLET PÁRTJA                                             39,119     31,495     70,614   0.7%    5%  ✗


In [38]:
# --- Nationality preferential mandate ---
# A nationality list wins a preferential seat if its votes >= (total_direct / 93) / 4
kedvezmenyes_kvota = total_direct / 93
quarter_kvota = kedvezmenyes_kvota / 4

print(f"Kedvezményes kvóta: {kedvezmenyes_kvota:,.0f}")
print(f"Quarter-kvóta (threshold for preferential seat): {quarter_kvota:,.0f}")
print()

nationality_seats = {}
for lista, v in nationality_list_votes.items():
    if v >= quarter_kvota:
        nationality_seats[lista] = 1
        print(f"  ✓ {lista}: {v:,} votes → wins preferential mandate")
    else:
        print(f"  ✗ {lista}: {v:,} votes")

n_nationality_seats = sum(nationality_seats.values())
n_party_list_seats = 93 - n_nationality_seats
print(f"\nNationality seats: {n_nationality_seats}")
print(f"Remaining party list seats for d'Hondt: {n_party_list_seats}")

Kedvezményes kvóta: 57,596
Quarter-kvóta (threshold for preferential seat): 14,399

  ✓ MAGYARORSZÁGI NÉMETEK ORSZÁGOS ÖNKORMÁNYZATA: 24,630 votes → wins preferential mandate
  ✗ ORSZÁGOS HORVÁT ÖNKORMÁNYZAT: 1,760 votes
  ✗ ORSZÁGOS SZLOVÁK ÖNKORMÁNYZAT: 1,208 votes
  ✗ ORSZÁGOS RUSZIN ÖNKORMÁNYZAT: 645 votes
  ✗ MAGYARORSZÁGI ROMÁNOK ORSZÁGOS ÖNKORMÁNYZATA: 526 votes
  ✗ SZERB ORSZÁGOS ÖNKORMÁNYZAT: 418 votes
  ✗ UKRÁN ORSZÁGOS ÖNKORMÁNYZAT: 396 votes
  ✗ ORSZÁGOS LENGYEL ÖNKORMÁNYZAT: 281 votes
  ✗ MAGYARORSZÁGI GÖRÖGÖK ORSZÁGOS ÖNKORMÁNYZATA: 232 votes
  ✗ ORSZÁGOS SZLOVÉN ÖNKORMÁNYZAT: 219 votes
  ✗ ORSZÁGOS ÖRMÉNY ÖNKORMÁNYZAT: 163 votes
  ✗ BOLGÁR ORSZÁGOS ÖNKORMÁNYZAT: 157 votes

Nationality seats: 1
Remaining party list seats for d'Hondt: 92


In [39]:
def dhondt(vote_totals: dict, n_seats: int) -> dict:
    """d'Hondt allocation: returns {party: seats}."""
    quotients = []
    for party, votes in vote_totals.items():
        for divisor in range(1, n_seats + 1):
            quotients.append((votes / divisor, party))
    quotients.sort(reverse=True)

    seats = defaultdict(int)
    for q, party in quotients[:n_seats]:
        seats[party] += 1
    return dict(seats)


# Filter to eligible lists (those passing the threshold)
eligible = {
    lista: list_totals[lista]
    for lista in party_list_votes.index
    if (party_list_votes[lista] / total_direct) >= THRESHOLD[lista]
}

print("Eligible lists for d'Hondt:")
for lista, v in sorted(eligible.items(), key=lambda x: -x[1]):
    print(f"  {lista[:60]:60s} {v:>10,}")

list_seats = dhondt(eligible, n_party_list_seats)

print(f"\nd'Hondt allocation ({n_party_list_seats} seats):")
for party in sorted(list_seats, key=list_seats.get, reverse=True):
    print(f"  {party[:60]:60s} {list_seats[party]:>3}")

Eligible lists for d'Hondt:
  FIDESZ - MAGYAR POLGÁRI SZÖVETSÉG-KERESZTÉNYDEMOKRATA NÉPPÁR  4,139,494
  DEMOKRATIKUS KOALÍCIÓ-JOBBIK MAGYARORSZÁGÉRT MOZGALOM-MOMENT  3,515,446
  MI HAZÁNK MOZGALOM                                              636,715

d'Hondt allocation (92 seats):
  FIDESZ - MAGYAR POLGÁRI SZÖVETSÉG-KERESZTÉNYDEMOKRATA NÉPPÁR  46
  DEMOKRATIKUS KOALÍCIÓ-JOBBIK MAGYARORSZÁGÉRT MOZGALOM-MOMENT  39
  MI HAZÁNK MOZGALOM                                             7


## 6. Final seat summary — comparison with official 2022 results

In [40]:
SHORT = {
    FIDESZ: "Fidesz-KDNP",
    EEGY: "Egységben",
    "MI HAZÁNK MOZGALOM": "Mi Hazánk",
    "MAGYAR KÉTFARKÚ KUTYA PÁRT": "MKKP",
    "MEGOLDÁS MOZGALOM": "Megoldás",
    "NORMÁLIS ÉLET PÁRTJA": "Normális Élet",
}

# Official 2022 declared results (valasztas.hu)
# Note: szavazókör protocol data gives 87/19 OEVK split; official is 88/18.
# One close district was corrected after protocols → cascades into list seats.
OFFICIAL = {
    FIDESZ: {"oevk": 88, "list": 47},
    EEGY: {"oevk": 18, "list": 38},
    "MI HAZÁNK MOZGALOM": {"oevk": 0, "list": 7},
}

all_parties = set(list(seat_counts.index) + list(list_seats.keys()) + list(nationality_seats.keys()))

print(f"{'Party':<20s} {'OEVK':>5s} {'List':>5s} {'Total':>6s}  │  {'Off.OEVK':>8s} {'Off.List':>8s} {'Off.Total':>9s}  │ {'Match?'}")
print("─" * 95)

total_calc = 0
for party in sorted(all_parties, key=lambda p: -(seat_counts.get(p, 0) + list_seats.get(p, 0) + nationality_seats.get(p, 0))):
    o = seat_counts.get(party, 0)
    l = list_seats.get(party, 0) + nationality_seats.get(party, 0)
    t = o + l
    total_calc += t
    short = SHORT.get(party, party[:20])

    if party in OFFICIAL:
        off = OFFICIAL[party]
        off_t = off["oevk"] + off["list"]
        match = "✓" if (o == off["oevk"] and l == off["list"]) else "✗"
        print(f"  {short:<18s} {o:>5} {l:>5} {t:>6}  │  {off['oevk']:>8} {off['list']:>8} {off_t:>9}  │  {match}")
    else:
        print(f"  {short:<18s} {o:>5} {l:>5} {t:>6}  │  {'—':>8} {'—':>8} {'—':>9}  │")

print("─" * 95)
print(f"  {'TOTAL':<18s} {seat_counts.sum():>5} {sum(list_seats.values()) + n_nationality_seats:>5} {total_calc:>6}")

Party                 OEVK  List  Total  │  Off.OEVK Off.List Off.Total  │ Match?
───────────────────────────────────────────────────────────────────────────────────────────────
  Fidesz-KDNP           87    46    133  │        88       47       135  │  ✗
  Egységben             19    39     58  │        18       38        56  │  ✗
  Mi Hazánk              0     7      7  │         0        7         7  │  ✓
  MAGYARORSZÁGI NÉMETE     0     1      1  │         —        —         —  │
───────────────────────────────────────────────────────────────────────────────────────────────
  TOTAL                106    93    199


## 7. Reusable seat calculator function

Packages the entire pipeline into a single function that takes vote data and returns seat allocations.

In [41]:
def seat_calculator(
    oevk_votes_by_district: dict[str, dict[str, int]],
    direct_list_votes: dict[str, int],
    list_meta: dict[str, dict],
    nationality_list_votes: dict[str, int] | None = None,
    total_list_seats: int = 93,
) -> dict:
    """
    Deterministic Hungarian parliamentary seat calculator.

    Parameters
    ----------
    oevk_votes_by_district : dict
        {district_id: {party: votes}} for each of the 106 OEVKs.
        Parties must use the same names as in direct_list_votes where applicable.
    direct_list_votes : dict
        {list_name: total_direct_votes} from the party-list ballot.
    list_meta : dict
        {list_name: {"threshold": float}} e.g. 0.05 / 0.10 / 0.15.
    nationality_list_votes : dict or None
        {nationality_list_name: votes}. If provided, preferential mandates
        are computed and subtracted from the party list seat pool.
    total_list_seats : int
        Total number of list seats (default 93).

    Returns
    -------
    dict with keys:
        "district_seats": {district_id: winning_party}
        "district_seat_counts": {party: n}
        "fractional_votes": {party: n}
        "list_totals": {party: direct + fractional}
        "eligible_lists": {party: total} (only those passing threshold)
        "list_seat_counts": {party: n}
        "nationality_seats": {nationality: n}
        "total_seats": {party: n}
    """
    parties_with_lists = set(direct_list_votes.keys())
    total_direct = sum(direct_list_votes.values())

    # --- OEVK seats + fractional votes ---
    district_winners = {}
    frac = defaultdict(int)

    for dist, votes_dict in oevk_votes_by_district.items():
        ranked = sorted(votes_dict.items(), key=lambda x: -x[1])
        winner_party, winner_v = ranked[0]
        runner_up_v = ranked[1][1] if len(ranked) > 1 else 0

        district_winners[dist] = winner_party

        surplus = winner_v - (runner_up_v + 1)
        if winner_party in parties_with_lists:
            frac[winner_party] += surplus

        for party, v in ranked[1:]:
            if party in parties_with_lists:
                frac[party] += v

    district_seat_counts = defaultdict(int)
    for p in district_winners.values():
        district_seat_counts[p] += 1

    # --- List totals = direct + fractional ---
    list_totals = {}
    for lista in direct_list_votes:
        list_totals[lista] = direct_list_votes[lista] + frac.get(lista, 0)

    # --- Threshold check (on direct list votes) ---
    eligible = {}
    for lista in direct_list_votes:
        thr = list_meta.get(lista, {}).get("threshold", 0.05)
        if direct_list_votes[lista] / total_direct >= thr:
            eligible[lista] = list_totals[lista]

    # --- Nationality preferential mandates ---
    nat_seats = {}
    n_nat = 0
    if nationality_list_votes:
        kvota = total_direct / total_list_seats
        for nat, v in nationality_list_votes.items():
            if v >= kvota / 4:
                nat_seats[nat] = 1
                n_nat += 1

    # --- d'Hondt ---
    n_party_seats = total_list_seats - n_nat
    list_seat_counts = dhondt(eligible, n_party_seats)

    # --- Totals ---
    all_parties = set(list(district_seat_counts.keys()) + list(list_seat_counts.keys()))
    total_seats = {}
    for p in all_parties:
        total_seats[p] = district_seat_counts.get(p, 0) + list_seat_counts.get(p, 0)
    for nat, n in nat_seats.items():
        total_seats[nat] = total_seats.get(nat, 0) + n

    return {
        "district_seats": district_winners,
        "district_seat_counts": dict(district_seat_counts),
        "fractional_votes": dict(frac),
        "list_totals": list_totals,
        "eligible_lists": eligible,
        "list_seat_counts": list_seat_counts,
        "nationality_seats": nat_seats,
        "total_seats": total_seats,
    }

In [42]:
# --- Build inputs from parsed data ---
oevk_dict = {}
for district in oevk_votes["district"].unique():
    dv = oevk_votes[oevk_votes["district"] == district]
    oevk_dict[district] = dict(zip(dv["party"], dv["votes"]))

direct_dict = party_list_votes.to_dict()

meta_dict = {lista: {"threshold": THRESHOLD[lista]} for lista in THRESHOLD}

nat_dict = nationality_list_votes.to_dict()

# --- Run the calculator ---
result = seat_calculator(oevk_dict, direct_dict, meta_dict, nat_dict)

print("=== Seat Calculator Output ===")
print(f"\nDistrict seats:")
for p in sorted(result["district_seat_counts"], key=result["district_seat_counts"].get, reverse=True):
    print(f"  {SHORT.get(p, p[:30]):<20s} {result['district_seat_counts'][p]:>3}")

print(f"\nList seats:")
for p in sorted(result["list_seat_counts"], key=result["list_seat_counts"].get, reverse=True):
    print(f"  {SHORT.get(p, p[:30]):<20s} {result['list_seat_counts'][p]:>3}")
for nat, n in result["nationality_seats"].items():
    print(f"  {nat[:30]:<20s} {n:>3}  (nationality)")

print(f"\nTotal seats:")
for p in sorted(result["total_seats"], key=result["total_seats"].get, reverse=True):
    short = SHORT.get(p, p[:30])
    print(f"  {short:<30s} {result['total_seats'][p]:>3}")
print(f"  {'TOTAL':<30s} {sum(result['total_seats'].values()):>3}")

=== Seat Calculator Output ===

District seats:
  Fidesz-KDNP           87
  Egységben             19

List seats:
  Fidesz-KDNP           46
  Egységben             39
  Mi Hazánk              7
  MAGYARORSZÁGI NÉMETEK ORSZÁGOS   1  (nationality)

Total seats:
  Fidesz-KDNP                    133
  Egységben                       58
  Mi Hazánk                        7
  MAGYARORSZÁGI NÉMETEK ORSZÁGOS   1
  TOTAL                          199


## 8. Diagnostics: fractional vote breakdown and marginal d'Hondt quotients

Helps verify the non-linearity: which quotients were close to the cutoff?

In [43]:
# Show the d'Hondt quotients around the seat boundary
eligible = result["eligible_lists"]
quotients = []
for party, votes in eligible.items():
    for divisor in range(1, n_party_list_seats + 5):
        quotients.append((votes / divisor, SHORT.get(party, party[:20]), divisor))
quotients.sort(reverse=True)

print(f"d'Hondt quotients around seat boundary (seats {n_party_list_seats - 3} to {n_party_list_seats + 2}):")
print(f"{'Rank':>5s}  {'Quotient':>12s}  {'Party':<15s}  {'Divisor':>7s}  {'Seat?'}")
print("-" * 55)
for i, (q, party, div) in enumerate(quotients[n_party_list_seats - 4 : n_party_list_seats + 3], n_party_list_seats - 3):
    marker = " ← cutoff" if i == n_party_list_seats else ""
    inside = "✓" if i <= n_party_list_seats else " "
    print(f"  {i:>3}   {q:>12,.1f}  {party:<15s}  {div:>7}   {inside}{marker}")

d'Hondt quotients around seat boundary (seats 89 to 94):
 Rank      Quotient  Party            Divisor  Seat?
-------------------------------------------------------
   89       91,988.8  Fidesz-KDNP           45   ✓
   90       90,959.3  Mi Hazánk              7   ✓
   91       90,139.6  Egységben             39   ✓
   92       89,989.0  Fidesz-KDNP           46   ✓ ← cutoff
   93       88,074.3  Fidesz-KDNP           47    
   94       87,886.1  Egységben             40    
   95       86,239.5  Fidesz-KDNP           48    


In [44]:
# Close OEVK races (margin < 2000)
close_races = []
for dist, votes_dict in oevk_dict.items():
    ranked = sorted(votes_dict.items(), key=lambda x: -x[1])
    margin = ranked[0][1] - ranked[1][1]
    if margin < 2000:
        close_races.append((dist, margin, SHORT.get(ranked[0][0], ranked[0][0][:15]), ranked[0][1], ranked[1][1]))

close_races.sort(key=lambda x: x[1])
print("Close OEVK races (margin < 2000):")
print(f"{'District':>10s}  {'Margin':>7s}  {'Winner':<15s}  {'W.votes':>8s}  {'RU.votes':>8s}")
print("-" * 55)
for dist, margin, winner, wv, rv in close_races:
    print(f"  {dist:>8s}  {margin:>7,}  {winner:<15s}  {wv:>8,}  {rv:>8,}")

Close OEVK races (margin < 2000):
  District   Margin  Winner            W.votes  RU.votes
-------------------------------------------------------
     01-14      381  Fidesz-KDNP        25,114    24,733
     01-13      468  Egységben          25,080    24,612
      05-2      492  Fidesz-KDNP        20,602    20,110
     01-15      780  Egységben          25,946    25,166
      07-4    1,053  Fidesz-KDNP        20,562    19,509
      02-1    1,153  Egységben          22,710    21,557
      14-2    1,378  Fidesz-KDNP        34,810    33,432
     01-12    1,410  Egységben          23,397    21,987
      16-1    1,448  Fidesz-KDNP        23,254    21,806
      14-1    1,521  Fidesz-KDNP        29,597    28,076
      02-2    1,672  Fidesz-KDNP        22,898    21,226
      01-9    1,708  Egységben          22,990    21,282
      14-8    1,883  Fidesz-KDNP        29,584    27,701
     01-16    1,976  Egységben          22,856    20,880
